# 📊 Phase 2 — Exploratory Data Analysis
**Ironhack Final Project · Retail Rental Analytics System**

> *Retailers destroy value by discounting stale inventory. A structured rental program converts dead stock into a recurring revenue stream — and the data proves it.*

---

**Analyses in this notebook:**
1. Database connection & schema overview
2. Inventory aging curves by category (log-normal distributions)
3. Sell-through rates — % of inventory crossing the 365-day threshold
4. Price depreciation over time on shelf
5. Rental revenue vs. markdown revenue comparison (hypothesis preview)
6. Late return rates by category
7. Key findings summary
8. Tableau exports


## 0. Setup & Configuration

In [ ]:
# --- Standard imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import lognorm, kstest, linregress
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# --- Plot style ---
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.titleweight': 'bold',
})

CATEGORY_PALETTE = sns.color_palette('tab10', 8)
print('✅ Imports OK')

In [ ]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM v_inventory_aging", engine)

In [ ]:
with engine.connect() as conn:
    tables = conn.execute(text('SHOW TABLES')).fetchall()

print(f'✅ Connected to {rental_final_project}')
print('Tables:', [t[0] for t in tables])

## 1. Schema Overview & Row Counts

In [ ]:
tables_of_interest = [
    'categories', 'products', 'inventory_events',
    'pricing_rules', 'customers', 'rentals', 'return_conditions'
]

counts = {}
with engine.connect() as conn:
    for t in tables_of_interest:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        counts[t] = n

df_counts = pd.DataFrame.from_dict(counts, orient='index', columns=['row_count'])
print('=== Row counts ===')
print(df_counts.to_string())

In [ ]:
# Load analytical views
with engine.connect() as conn:
    df_aging    = pd.read_sql('SELECT * FROM v_inventory_aging',    conn)
    df_eligible = pd.read_sql('SELECT * FROM v_rental_eligible',    conn)
    df_history  = pd.read_sql('SELECT * FROM v_rental_history',     conn)
    df_revenue  = pd.read_sql('SELECT * FROM v_revenue_comparison', conn)

# Load core tables
with engine.connect() as conn:
    df_products = pd.read_sql('SELECT * FROM products', conn)
    df_cats     = pd.read_sql('SELECT * FROM categories', conn)
    df_rentals  = pd.read_sql('SELECT * FROM rentals', conn)
    df_returns  = pd.read_sql('SELECT * FROM return_conditions', conn)
    df_pricing  = pd.read_sql('SELECT * FROM pricing_rules', conn)

df_products = df_products.merge(df_cats[['category_id','category_name']], on='category_id', how='left')
print('Core tables loaded ✅')
df_products.head(3)

---
## 2. Inventory Aging Curves by Category

**Question:** How long does inventory sit before being rented or marked down? Which categories have the heaviest tails past 365 days?

We expect log-normal distributions of `days_on_shelf`. The 365-day vertical line marks rental eligibility.

In [ ]:
# Build days_on_shelf from inventory_events (first 'listed' event per product)
query = """
SELECT
    p.product_id,
    c.category_name,
    DATEDIFF(CURDATE(), ie.event_date) AS days_on_shelf,
    p.retail_price
FROM products p
JOIN categories c        ON p.category_id  = c.category_id
JOIN inventory_events ie ON p.product_id   = ie.product_id
WHERE ie.event_type = 'listed'
  AND ie.event_id = (
      SELECT MIN(ie2.event_id)
      FROM inventory_events ie2
      WHERE ie2.product_id = p.product_id
        AND ie2.event_type = 'listed'
  )
"""
with engine.connect() as conn:
    df_shelf = pd.read_sql(text(query), conn)

# Fallback: use the aging view if the query above returns nothing
if df_shelf.empty and 'days_on_shelf' in df_aging.columns:
    df_shelf = df_aging.copy()

print(f'{len(df_shelf):,} products with shelf-age data')
df_shelf.describe()

In [ ]:
# --- 2a. KDE aging curves by category ---
cats = df_shelf['category_name'].dropna().unique()
ncols = 4
nrows = -(-len(cats) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for ax, (cat, colour) in zip(axes, zip(cats, CATEGORY_PALETTE)):
    sub = df_shelf[df_shelf['category_name'] == cat]['days_on_shelf'].dropna()
    sns.histplot(sub, bins=40, kde=True, ax=ax, color=colour, alpha=0.5, stat='density')
    ax.axvline(365, color='crimson', ls='--', lw=1.5, label='365-day threshold')
    ax.axvline(sub.median(), color='navy', ls=':', lw=1.5, label=f'Median: {sub.median():.0f}d')
    pct_stale = (sub > 365).mean() * 100
    ax.set_title(f'{cat}\n{pct_stale:.1f}% past 365 days')
    ax.set_xlabel('Days on shelf')
    ax.legend(fontsize=8)

for ax in axes[len(cats):]:
    ax.set_visible(False)

fig.suptitle('Inventory Aging Distribution by Category', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('aging_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Saved aging_curves.png')

In [ ]:
# --- 2b. Log-normal fit quality table ---
lognorm_summary = []
for cat in cats:
    sub = df_shelf[df_shelf['category_name'] == cat]['days_on_shelf'].dropna()
    sub_pos = sub[sub > 0]
    shape, loc, scale = lognorm.fit(sub_pos, floc=0)
    stat, pval = kstest(sub_pos, 'lognorm', args=(shape, loc, scale))
    lognorm_summary.append({
        'category':        cat,
        'n':               len(sub_pos),
        'median_days':     sub.median(),
        'pct_over_365':    (sub > 365).mean() * 100,
        'log_sigma':       shape,
        'log_mu':          np.log(scale),
        'KS_stat':         stat,
        'KS_pvalue':       pval,
        'lognormal_fit':   'good' if pval > 0.05 else 'marginal'
    })

df_lognorm = pd.DataFrame(lognorm_summary).sort_values('pct_over_365', ascending=False)
df_lognorm.style.background_gradient(subset=['pct_over_365'], cmap='YlOrRd').format({
    'pct_over_365': '{:.1f}%', 'median_days': '{:.0f}',
    'log_mu': '{:.2f}', 'log_sigma': '{:.2f}',
    'KS_stat': '{:.3f}', 'KS_pvalue': '{:.3f}'
})

---
## 3. Sell-Through Rates — The "Dead Zone"

**Question:** What fraction of each category's inventory crosses the 365-day threshold? This defines the rental-eligible pool and is the core commercial opportunity.

In [ ]:
query_st = """
SELECT
    c.category_name,
    COUNT(*) AS total_products,
    SUM(CASE WHEN DATEDIFF(CURDATE(), ie_list.event_date) <= 365 THEN 1 ELSE 0 END) AS sold_within_365,
    SUM(CASE WHEN DATEDIFF(CURDATE(), ie_list.event_date) >  365 THEN 1 ELSE 0 END) AS stale_over_365
FROM products p
JOIN categories c ON p.category_id = c.category_id
JOIN (
    SELECT product_id, MIN(event_date) AS event_date
    FROM inventory_events
    WHERE event_type = 'listed'
    GROUP BY product_id
) ie_list ON p.product_id = ie_list.product_id
GROUP BY c.category_name
ORDER BY stale_over_365 DESC
"""
with engine.connect() as conn:
    df_st = pd.read_sql(text(query_st), conn)

df_st['sell_through_rate'] = df_st['sold_within_365'] / df_st['total_products'] * 100
df_st['stale_rate']        = df_st['stale_over_365']  / df_st['total_products'] * 100
df_st

In [ ]:
# --- 3a. Stacked bar: sold within 365d vs. stale ---
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(df_st))
w = 0.6

bars_sold  = ax.bar(x, df_st['sell_through_rate'], w, label='Sold within 365 days', color='#2ecc71')
bars_stale = ax.bar(x, df_st['stale_rate'], w, bottom=df_st['sell_through_rate'],
                    label='Stale (>365 days) — rental eligible', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(df_st['category_name'], rotation=30, ha='right')
ax.set_ylabel('% of inventory')
ax.set_title('Sell-Through Rate vs. Stale Inventory by Category\n(items crossing 365-day threshold enter the rental pool)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()

for rect, row in zip(bars_stale, df_st.itertuples()):
    ax.text(rect.get_x() + rect.get_width()/2,
            row.sell_through_rate + row.stale_rate/2,
            f'{row.stale_rate:.1f}%', ha='center', va='center',
            color='white', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('sell_through.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 3b. Stale retail value at risk ---
query_value = """
SELECT
    c.category_name,
    SUM(p.retail_price) AS total_retail_value,
    SUM(CASE WHEN DATEDIFF(CURDATE(), ie_list.event_date) > 365 THEN p.retail_price ELSE 0 END) AS stale_retail_value,
    COUNT(CASE WHEN DATEDIFF(CURDATE(), ie_list.event_date) > 365 THEN 1 END) AS stale_units
FROM products p
JOIN categories c ON p.category_id = c.category_id
JOIN (
    SELECT product_id, MIN(event_date) AS event_date
    FROM inventory_events WHERE event_type = 'listed'
    GROUP BY product_id
) ie_list ON p.product_id = ie_list.product_id
GROUP BY c.category_name
ORDER BY stale_retail_value DESC
"""
with engine.connect() as conn:
    df_value = pd.read_sql(text(query_value), conn)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.barh(df_value['category_name'], df_value['stale_retail_value'],
               color=CATEGORY_PALETTE)
ax.set_xlabel('Retail value at risk (€)')
ax.set_title('Stale Inventory Value by Category\n(potential rental revenue vs. markdown losses)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:,.0f}'))

for bar, row in zip(bars, df_value.itertuples()):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
            f'{row.stale_units} units', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('stale_value.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 4. Price Depreciation Over Time

**Question:** How does retail price correlate with days on shelf? Does price decline as items age — and how does that differ by category? This baseline underpins the markdown revenue side of the comparison.

In [ ]:
# Merge shelf age into products for depreciation analysis
df_dep = df_shelf.merge(
    df_products[['product_id','retail_price']].rename(columns={'retail_price': 'retail_price_prod'}),
    on='product_id', how='left'
)
# Use retail_price from shelf query if available, else fallback
df_dep['retail_price'] = df_dep.get('retail_price', df_dep.get('retail_price_prod'))

df_dep['age_bucket'] = pd.cut(
    df_dep['days_on_shelf'],
    bins=[0, 90, 180, 270, 365, 548, 730, 99999],
    labels=['0–90d', '91–180d', '181–270d', '271–365d', '1–1.5yr', '1.5–2yr', '2yr+']
)
df_dep.head(3)

In [ ]:
# --- 4a. Scatter: price vs. days on shelf ---
fig, ax = plt.subplots(figsize=(13, 6))

for cat, colour in zip(cats, CATEGORY_PALETTE):
    sub = df_dep[df_dep['category_name'] == cat]
    ax.scatter(sub['days_on_shelf'], sub['retail_price'],
               alpha=0.25, s=18, color=colour, label=cat)

clean = df_dep.dropna(subset=['days_on_shelf', 'retail_price'])
slope, intercept, r, p, se = linregress(clean['days_on_shelf'], clean['retail_price'])
x_line = np.linspace(0, clean['days_on_shelf'].max(), 200)
ax.plot(x_line, intercept + slope * x_line, 'k--', lw=2,
        label=f'OLS trend (r={r:.2f}, slope={slope:.3f}€/day)')
ax.axvline(365, color='crimson', ls='--', lw=1.5, label='365-day threshold')

ax.set_xlabel('Days on shelf')
ax.set_ylabel('Retail price (€)')
ax.set_title('Price Depreciation: Retail Price vs. Days on Shelf')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('price_depreciation.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 4b. Box: retail price by age bucket ---
order = ['0–90d', '91–180d', '181–270d', '271–365d', '1–1.5yr', '1.5–2yr', '2yr+']
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=df_dep, x='age_bucket', y='retail_price', order=order,
            palette='coolwarm_r', ax=ax, showfliers=False)
ax.set_xlabel('Shelf-age bucket')
ax.set_ylabel('Retail price (€)')
ax.set_title('Median Retail Price by Shelf-Age Cohort')
plt.tight_layout()
plt.savefig('price_by_age_bucket.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 4c. Depreciation slope per category ---
dep_slopes = []
for cat in cats:
    sub = df_dep[df_dep['category_name'] == cat].dropna(subset=['days_on_shelf', 'retail_price'])
    if len(sub) < 10:
        continue
    s, i, r, p, se = linregress(sub['days_on_shelf'], sub['retail_price'])
    dep_slopes.append({'category': cat, 'slope_eur_per_day': s, 'r_squared': r**2, 'p_value': p})

df_slopes = pd.DataFrame(dep_slopes).sort_values('slope_eur_per_day')

fig, ax = plt.subplots(figsize=(10, 5))
colours = ['#e74c3c' if s < 0 else '#2ecc71' for s in df_slopes['slope_eur_per_day']]
ax.barh(df_slopes['category'], df_slopes['slope_eur_per_day'], color=colours)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Price change per additional day on shelf (€)')
ax.set_title('Price Depreciation Rate by Category')
plt.tight_layout()
plt.savefig('depreciation_by_category.png', bbox_inches='tight', dpi=150)
plt.show()
df_slopes

---
## 5. Rental Revenue vs. Markdown Revenue — Hypothesis Preview

**Central claim:** For items that entered the rental program, did they generate more revenue than a 50% markdown sale would have?

Phase 3 will validate this with Monte Carlo simulation. Here we do the first-look EDA.

In [ ]:
# Build revenue comparison (or use v_revenue_comparison view)
if df_revenue.empty:
    query_rev = """
    SELECT
        r.rental_id,
        c.category_name,
        p.retail_price,
        p.retail_price * 0.50                                       AS markdown_revenue,
        (DATEDIFF(r.return_date, r.rental_date) * r.daily_rate)
          + r.deposit_paid                                          AS rental_revenue,
        DATEDIFF(r.return_date, r.rental_date)                      AS rental_days
    FROM rentals r
    JOIN products p   ON r.product_id  = p.product_id
    JOIN categories c ON p.category_id = c.category_id
    WHERE r.return_date IS NOT NULL
    """
    with engine.connect() as conn:
        df_rev = pd.read_sql(text(query_rev), conn)
else:
    df_rev = df_revenue.copy()

df_rev['revenue_delta'] = df_rev['rental_revenue'] - df_rev['markdown_revenue']
df_rev['rental_wins']   = df_rev['revenue_delta'] > 0

print(f'Completed rentals  : {len(df_rev):,}')
print(f'Rental beats 50% markdown: {df_rev["rental_wins"].sum():,}  ({df_rev["rental_wins"].mean()*100:.1f}%)')
print(f'Mean revenue uplift      : €{df_rev["revenue_delta"].mean():.2f}')

In [ ]:
# --- 5a. Side-by-side box plots by category ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cat_order = (df_rev.groupby('category_name')['rental_revenue'].median()
             .sort_values(ascending=False).index.tolist())

sns.boxplot(data=df_rev, x='category_name', y='rental_revenue', order=cat_order,
            palette='Blues_d', ax=axes[0], showfliers=False)
axes[0].set_title('Rental Revenue per Transaction')
axes[0].set_xlabel('')
axes[0].set_ylabel('Revenue (€)')
axes[0].tick_params(axis='x', rotation=35)

sns.boxplot(data=df_rev, x='category_name', y='markdown_revenue', order=cat_order,
            palette='Reds_d', ax=axes[1], showfliers=False)
axes[1].set_title('Equivalent 50% Markdown Revenue')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue (€)')
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('Rental vs. Markdown Revenue — First Look', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('rental_vs_markdown.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 5b. Revenue uplift % by category ---
agg = df_rev.groupby('category_name').agg(
    mean_rental_rev   = ('rental_revenue',   'mean'),
    mean_markdown_rev = ('markdown_revenue',  'mean'),
    pct_rental_wins   = ('rental_wins',       'mean'),
    n                 = ('rental_id',         'count')
).reset_index()
agg['uplift_pct'] = (agg['mean_rental_rev'] / agg['mean_markdown_rev'] - 1) * 100
agg = agg.sort_values('uplift_pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colours = ['#2ecc71' if v > 0 else '#e74c3c' for v in agg['uplift_pct']]
ax.bar(agg['category_name'], agg['uplift_pct'], color=colours)
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Revenue uplift vs. 50% markdown (%)')
ax.set_title('Average Rental Revenue Uplift over Markdown by Category')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('revenue_uplift.png', bbox_inches='tight', dpi=150)
plt.show()

agg[['category_name','mean_rental_rev','mean_markdown_rev','uplift_pct','pct_rental_wins','n']]

In [ ]:
# --- 5c. Rental duration distribution (seeds Phase 3 Weibull model) ---
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(df_rev['rental_days'].dropna(), bins=50, kde=True, ax=ax, color='steelblue')
ax.axvline(df_rev['rental_days'].median(), color='navy', ls='--', lw=2,
           label=f'Median: {df_rev["rental_days"].median():.0f} days')
ax.set_xlabel('Rental duration (days)')
ax.set_title('Distribution of Rental Durations\n(key input for Phase 3 Weibull survival model)')
ax.legend()
plt.tight_layout()
plt.savefig('rental_duration.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 6. Late Return Rates by Category

**Question:** Which categories see the most late returns? This is the primary signal for the Phase 4 customer churn risk classifier.

In [ ]:
query_late = """
SELECT
    r.rental_id,
    c.category_name,
    r.rental_date,
    r.due_date,
    r.return_date,
    rc.condition_grade,
    DATEDIFF(r.return_date, r.due_date) AS days_late,
    CASE WHEN r.return_date > r.due_date THEN 1 ELSE 0 END AS is_late
FROM rentals r
JOIN products p    ON r.product_id  = p.product_id
JOIN categories c  ON p.category_id = c.category_id
LEFT JOIN return_conditions rc ON r.rental_id = rc.rental_id
WHERE r.return_date IS NOT NULL
"""
with engine.connect() as conn:
    df_late = pd.read_sql(text(query_late), conn)

print(f'Completed rentals: {len(df_late):,}')
print(f'Late returns     : {df_late["is_late"].sum():,}  ({df_late["is_late"].mean()*100:.1f}%)')

In [ ]:
# --- 6a. Late return rate by category ---
late_by_cat = (df_late.groupby('category_name')['is_late']
               .agg(['sum','mean','count'])
               .rename(columns={'sum':'late_count','mean':'late_rate','count':'total'})
               .sort_values('late_rate', ascending=False)
               .reset_index())

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(late_by_cat['category_name'], late_by_cat['late_rate'] * 100,
              color=[CATEGORY_PALETTE[i % 8] for i in range(len(late_by_cat))])
ax.set_ylabel('Late return rate (%)')
ax.set_title('Late Return Rate by Category\n(returned after due_date)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=30, ha='right')

for bar, row in zip(bars, late_by_cat.itertuples()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'n={row.total}', ha='center', fontsize=8, color='grey')

plt.tight_layout()
plt.savefig('late_return_rate.png', bbox_inches='tight', dpi=150)
plt.show()
late_by_cat

In [ ]:
# --- 6b. Days overdue distribution (late returns only) ---
late_only = df_late[df_late['is_late'] == 1].copy()

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(late_only['days_late'], bins=40, kde=True, ax=ax, color='tomato')
ax.axvline(late_only['days_late'].median(), color='darkred', ls='--', lw=2,
           label=f'Median: {late_only["days_late"].median():.0f} days overdue')
ax.set_xlabel('Days overdue')
ax.set_title('How Late Are Late Returns?\n(feeds Phase 4 customer churn risk classifier)')
ax.legend()
plt.tight_layout()
plt.savefig('days_overdue.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# --- 6c. Return condition vs. timeliness (if condition_grade is populated) ---
if 'condition_grade' in df_late.columns and df_late['condition_grade'].notna().any():
    grade_order = sorted(df_late['condition_grade'].dropna().unique())
    crosstab = pd.crosstab(df_late['condition_grade'], df_late['is_late'],
                           normalize='index') * 100
    crosstab.columns = ['On time', 'Late']
    crosstab = crosstab.reindex([g for g in grade_order if g in crosstab.index])

    fig, ax = plt.subplots(figsize=(10, 5))
    crosstab.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71','#e74c3c'])
    ax.set_ylabel('% of returns')
    ax.set_title('Return Condition Grade vs. Timeliness\n(do late returns come back in worse condition?)')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('condition_vs_lateness.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print('ℹ️  condition_grade not populated — skipping chart 6c')

---
## 7. Key Findings Summary

In [ ]:
worst_cat     = df_lognorm.iloc[0]
most_value    = df_value.iloc[0] if not df_value.empty else None
uplift_winner = agg.iloc[0]
late_leader   = late_by_cat.iloc[0]
overall_stale = df_st['stale_over_365'].sum() / df_st['total_products'].sum() * 100

print('=' * 60)
print('📊 PHASE 2 KEY FINDINGS')
print('=' * 60)
print(f"""
1. INVENTORY AGING
   • '{worst_cat['category']}' has the highest stale rate:
     {worst_cat['pct_over_365']:.1f}% of products cross the 365-day threshold.
   • Median shelf time across all categories: {df_shelf['days_on_shelf'].median():.0f} days.

2. SELL-THROUGH / STALE INVENTORY
   • {overall_stale:.1f}% of total inventory is rental-eligible (>365 days on shelf).
   {'• Highest stale value at risk: ' + most_value['category_name'] + ' (€' + f"{most_value['stale_retail_value']:,.0f}" + ')' if most_value is not None else ''}

3. PRICE DEPRECIATION
   • Overall OLS slope: €{slope:.3f}/day  (r={r:.2f})
   • Category-level variation suggests depreciation driven by product mix.

4. RENTAL vs. MARKDOWN REVENUE
   • {df_rev['rental_wins'].mean()*100:.1f}% of completed rentals beat a 50% markdown.
   • '{uplift_winner['category_name']}' shows highest uplift: +{uplift_winner['uplift_pct']:.1f}% over markdown.
   • Hypothesis SUPPORTED — full validation in Phase 3 Monte Carlo.

5. LATE RETURN RATES
   • '{late_leader['category_name']}' has the highest late rate: {late_leader['late_rate']*100:.1f}%.
   • Overall late return rate: {df_late['is_late'].mean()*100:.1f}%.
   • Key input for Phase 4 customer churn risk classifier.
""")
print('=' * 60)
print('→ Next: Phase 3 — Monte Carlo revenue simulation & Weibull survival model')

---
## 8. Export Summary Stats for Tableau Dashboard 1

In [ ]:
import os
os.makedirs('tableau_exports', exist_ok=True)

df_shelf.to_csv('tableau_exports/inventory_aging.csv', index=False)
df_st.to_csv('tableau_exports/sell_through.csv', index=False)
df_value.to_csv('tableau_exports/stale_value_by_category.csv', index=False)
df_dep.to_csv('tableau_exports/price_depreciation.csv', index=False)
df_rev.to_csv('tableau_exports/rental_vs_markdown.csv', index=False)
late_by_cat.to_csv('tableau_exports/late_return_rates.csv', index=False)
df_lognorm.to_csv('tableau_exports/lognorm_fit_summary.csv', index=False)

print('✅ All Tableau CSVs written to ./tableau_exports/')
print('   Feed these into Tableau Dashboard 1: Inventory Health Overview')